In [1]:
import pandas as pd

path="olist_data/"

customers=pd.read_csv(path+"olist_customers_dataset.csv")
orders=pd.read_csv(path+"olist_orders_dataset.csv")
items=pd.read_csv(path+"olist_order_items_dataset.csv")
products=pd.read_csv(path+"olist_products_dataset.csv")
category_translate=pd.read_csv(path+"product_category_name_translation.csv")
geoloc=pd.read_csv(path+"olist_geolocation_dataset.csv")
payments=pd.read_csv(path+"olist_order_payments_dataset.csv")
reviews=pd.read_csv(path+"olist_order_reviews_dataset.csv")
seller=pd.read_csv(path+"olist_sellers_dataset.csv")


In [2]:
for name, df in {"orders":orders,"items":items,"reviews":reviews,
                 "products":products,"payments":payments,"seller":seller,
                 "customers":customers,"category_translate":
                 category_translate, "geoloc":geoloc}.items():
    print(name,df.shape)
    print(df.dtypes)
    print(df.isna().sum())
    print("~"*40)

orders (99441, 8)
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
items (112650, 7)
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
order_id               0
order_item_id          0
product_id     

In [12]:
#FIXING DATA TYPES
date_cols=["order_purchase_timestamp","order_approved_at",
              "order_delivered_carrier_date","order_delivered_customer_date",
              "order_estimated_delivery_date"]
for col in date_cols:
    orders[col]=pd.to_datetime(orders[col],errors="coerce")

In [ ]:
#Orders that are not delivered 
orders["is_delivered"] = orders["order_delivered_customer_date"].notnull()


In [14]:
#Reviews with no written comments that are legitimate but not required
reviews["review_comment_message"]=reviews["review_comment_message"].fillna(" ")

In [5]:
#Missing Product category
products["product_category_name"]=products["product_category_name"].fillna("unknown")


In [3]:

#Duplicate values
print(orders.duplicated(subset="order_id").sum())
print(customers.duplicated(subset="customer_id").sum())
orders=orders.drop_duplicates(subset="order_id")


0
0


In [15]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

In [16]:
#translate product category
products=products.merge(category_translate,how="left",on="product_category_name")

In [17]:
#EDA

#Distribution of Review Score
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [18]:
#Distribution of Delivery Delays
orders["delivery_delay_days"].describe

<bound method NDFrame.describe of 0        -8.0
1        -6.0
2       -18.0
3       -13.0
4       -10.0
         ... 
99436   -11.0
99437    -2.0
99438    -6.0
99439   -21.0
99440   -18.0
Name: delivery_delay_days, Length: 99441, dtype: float64>

In [20]:
# Orders Delivered very late(Outliers)
orders[orders["delivery_delay_days"]>=30].shape[0]

360

In [21]:
#Correlation of delay with review score
merged=orders.merge(reviews,on="order_id")
merged[["delivery_delay_days","review_score"]].corr()

,delivery_delay_days,review_score
delivery_delay_days,1.000000,-0.266764
review_score,-0.266764,1.000000


In [23]:
path="clean/"
orders.to_csv(path+"cleaned_orders.csv",index=False)
items.to_csv(path+"cleaned_items.csv",index=False)
reviews.to_csv(path+"cleaned_reviews.csv",index=False)
products.to_csv(path+"cleaned_products.csv",index=False)
payments.to_csv(path+"cleaned_payments.csv",index=False)
seller.to_csv(path+"cleaned_seller.csv",index=False)
customers.to_csv(path+"cleaned_customers.csv",index=False)
category_translate.to_csv(path+"cleaned_category_translate.csv",index=False)
geoloc.to_csv(path+"cleaned_geoloc.csv",index=False)